In [ ]:
# Enable CUDA GPU - SELECT GPU T4 ×2 ON KAGGLE!
import os
import sys

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("🚀 CUDA GPU Setup...")
    # Detect CUDA version and install matching CuPy
    try:
        import subprocess
        nvcc_output = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
        if 'release 12' in nvcc_output.stdout:
            print("✓ Detected CUDA 12.x")
            !pip uninstall -y cupy-cuda11x cupy-cuda12x cupy -q 2>/dev/null || true
            !pip install -q cupy-cuda12x
        else:
            print("✓ Detected CUDA 11.x")
            !pip uninstall -y cupy-cuda11x cupy-cuda12x cupy -q 2>/dev/null || true
            !pip install -q cupy-cuda11x
    except:
        # Default to CUDA 12 (Kaggle standard)
        print("✓ Defaulting to CUDA 12.x")
        !pip uninstall -y cupy-cuda11x cupy-cuda12x cupy -q 2>/dev/null || true
        !pip install -q cupy-cuda12x
    
    try:
        import cupy as cp
        device = cp.cuda.Device()
        gpu_name = cp.cuda.runtime.getDeviceProperties(device.id)['name'].decode('utf-8')
        total_mem, free_mem = cp.cuda.Device().mem_info
        print(f"✓ GPU: {gpu_name}, Memory: {total_mem / 1e9:.1f} GB")
        USE_GPU = True
    except Exception as e:
        print(f"⚠️ GPU setup failed: {e}")
        USE_GPU = False
else:
    print("Local CPU mode")
    USE_GPU = False

print("="*60)

In [ ]:
# Clone Repository
import os, sys

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    if not os.path.exists('/kaggle/working/NLP_PROJECT_2025'):
        !git clone -q -b mohab https://github.com/MohabYasser2/NLP_PROJECT_2025.git /kaggle/working/NLP_PROJECT_2025
    else:
        !cd /kaggle/working/NLP_PROJECT_2025 && git fetch -q origin mohab && git reset --hard origin/mohab
    
    # Clear old cache
    !rm -f /kaggle/working/*_processed.pkl
    sys.path.insert(0, '/kaggle/working/NLP_PROJECT_2025')
    print("✓ Repo ready")
else:
    sys.path.insert(0, os.path.abspath('..'))

In [ ]:
# Detect Dataset Paths
import glob

if IS_KAGGLE:
    data_files = glob.glob('/kaggle/input/**/*.txt', recursive=True)
    if data_files:
        dataset_dir = os.path.dirname(data_files[0])
        TRAIN_FILE = os.path.join(dataset_dir, 'train.txt')
        DEV_FILE = os.path.join(dataset_dir, 'val.txt') if 'val.txt' in str(data_files) else os.path.join(dataset_dir, 'dev.txt')
        TEST_FILE = os.path.join(dataset_dir, 'test.txt')
    else:
        print("⚠️ No dataset found! Add your dataset in Kaggle.")
else:
    TRAIN_FILE = '../data/train.txt'
    DEV_FILE = '../data/val.txt'
    TEST_FILE = '../data/test.txt'

OUTPUT_FILE = '/kaggle/working/submission.csv' if IS_KAGGLE else 'submission.csv'
print(f"✓ Train: {TRAIN_FILE}")

In [ ]:
# TRAIN LOGISTIC REGRESSION MODEL
import pickle
import numpy as np
from pathlib import Path
from src.training.train_logreg import run_logreg_training

MODEL_PATH = '/kaggle/working/NLP_PROJECT_2025/models/logreg_model.pkl' if IS_KAGGLE else '../models/logreg_model.pkl'

if os.path.exists(MODEL_PATH):
    print("="*70)
    print("📦 LOADING PRE-TRAINED MODEL")
    print("="*70)
    
    from src.models.logreg_model import LogisticRegressionModel
    from src.preprocessing import prepare_dataset
    
    with open(MODEL_PATH, 'rb') as f:
        model = pickle.load(f)
    
    # Load dev data for evaluation
    cache_dir = Path('/kaggle/working') if IS_KAGGLE else Path('../data')
    cache_file = cache_dir / 'val_processed.pkl'
    
    if cache_file.exists():
        with open(cache_file, 'rb') as f:
            data = pickle.load(f)
        dev_texts, dev_labels = data['texts'], data['labels']
    else:
        from src.preprocessing import prepare_dataset
        dev_texts, dev_labels = prepare_dataset(str(DEV_FILE), str(cache_file.with_suffix('')))
    
    # Evaluate
    metrics = model.evaluate(dev_texts, dev_labels, window_size=5)
    print(f"\n✓ Accuracy: {metrics['accuracy']*100:.2f}%")
    print(f"✓ DER: {metrics['der']*100:.2f}%")
    print("="*70)
    
else:
    print("="*70)
    print("🚀 TRAINING LOGISTIC REGRESSION")
    print("="*70)
    print("\n📋 Configuration:")
    print("   • Features: 3,000 (minimal for <30GB RAM)")
    print("   • N-grams: 1-2 (character bigrams)")
    print("   • Window: 5, LR: 0.1, Epochs: 100")
    print("   • Streaming: Mini-batch training (500 sentences/chunk)")
    print("\n⏱️  Expected: ~10-15 min, Peak RAM: ~12-18GB")
    print("="*70)
    
    # Call the training pipeline (all logic is in train_logreg.py)
    run_logreg_training(
        train_file=TRAIN_FILE,
        dev_file=DEV_FILE,
        test_file=None,
        output_file=None
    )
    
    print("\n" + "="*70)
    print("✅ TRAINING COMPLETE")
    print("="*70)
    print(f"✓ Model saved to: {MODEL_PATH}")